In [32]:
import ctypes
import nanoarrow as na
import numpy as np
import pandas as pd
import pyarrow as pa
print(pd.__version__)
print(pa.__version__)

3.0.3
24.0.0


# NumPy to PyArrow

## Primitive data types (integer, float, ...)

Primitive, one dimensional ndarray, no missing

In [33]:
np_arr_p1 = np.array([1,0,3])
pa_arr_p1 = pa.array(np_arr_p1)
pa_arr_p1

[
  1,
  0,
  3
]

Memory is shared

In [34]:
pa_arr_p1.buffers()[1].address == np_arr_p1.ctypes.data

True

What about primitive with np.nan?

In [35]:
np_arr_p2 = np.array([1,0,np.nan])
pa_arr_p2 = pa.array(np_arr_p2)
pa_arr_p2

[
  1,
  0,
  nan
]

In [36]:
pa_arr_p2.buffers()[1].address == np_arr_p2.ctypes.data

True

Memory can still be shared. What if we need validity bitmap instead of sentinel value?

In [37]:
np_arr_p3 = np.array([1,0,np.nan])
pa_arr_p3 = pa.array(np_arr_p3, from_pandas=True)
pa_arr_p3

[
  1,
  0,
  null
]

In [38]:
pa_arr_p3.buffers()[1].address == np_arr_p3.ctypes.data

True

The data can still be shared, but in this case another buffer is allocated for the validity bitmap.

In [39]:
pa_arr_p3.buffers()

[<pyarrow.Buffer address=0x116860540 size=1 is_cpu=True is_mutable=True>,
 <pyarrow.Buffer address=0x945034020 size=24 is_cpu=True is_mutable=True>]

Whereas before, this was not needed:

In [40]:
pa_arr_p2.buffers()

[None,
 <pyarrow.Buffer address=0x945034060 size=24 is_cpu=True is_mutable=True>]

## Primitive Pandas NumpyBlock

In [41]:
df = pd.DataFrame({
    "id": [1, 2, 3, 4],
    "Cost": [550, 540, 200, 690],
})
b = pa.record_batch(df)
b

pyarrow.RecordBatch
id: int64
Cost: int64
----
id: [1,2,3,4]
Cost: [550,540,200,690]

In [42]:
df._mgr.blocks[0].values

array([[  1,   2,   3,   4],
       [550, 540, 200, 690]])

In [43]:
for c in b.columns:
    print(c.buffers()[1].address)

39825329088
39825329120


In [44]:
row0_addr = df._mgr.blocks[0].values[0, :].ctypes.data
row1_addr = df._mgr.blocks[0].values[1, :].ctypes.data
print("row0 addr:", row0_addr)
print("row1 addr:", row1_addr)

row0 addr: 39825329088
row1 addr: 39825329120


The data can be shared because of it being contiguous in memory. That is not necessarily the case if we have view of the numpy ndarray.

In [45]:
# Not contiguous
col = df._mgr.blocks[0].values[:, 1]
col.strides != col[0].itemsize  # a simple check for non-contiguous memory

True

In [46]:
col.flags

  C_CONTIGUOUS : False
  F_CONTIGUOUS : False
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

In [47]:
# for non-contiguous data the conversion is not zero-copy
arr_col = pa.array(col)
arr_col.buffers()[1].address != col.ctypes.data

True

In [48]:
arr_col

[
  2,
  540
]

In [49]:
col

array([  2, 540])

## Pandas ExtensionBlock (1-D)

In [50]:
df2 = pd.DataFrame({
    "id": pd.array([1, 2, pd.NA]),
    "count": pd.array([1, 2, 3])
})
df2.dtypes

id       Int64
count    Int64
dtype: object

In [51]:
b2 = pa.record_batch(df2)
b2

pyarrow.RecordBatch
id: int64
count: int64
----
id: [1,2,null]
count: [1,2,3]

In [52]:
df2._mgr

BlockManager
Items: Index(['id', 'count'], dtype='object')
Axis 1: RangeIndex(start=0, stop=3, step=1)
ExtensionBlock: slice(0, 1, 1), 1 x 3, dtype: Int64
ExtensionBlock: slice(1, 2, 1), 1 x 3, dtype: Int64

In [53]:
df2._mgr.blocks[0].values._data

array([1, 2, 1])

In [54]:
df2._mgr.blocks[0].values._mask

array([False, False,  True])

Data values can be zero-copied

In [55]:
df2._mgr.blocks[0].values._data.ctypes.data == b2.column(0).buffers()[1].address

True

But the validity bitmap/mask can't be

In [56]:
df2._mgr.blocks[0].values._mask.ctypes.data == b2.column(0).buffers()[0].address

False

## Boolean data type is always copied

In pandas/numoy, one boolean element occupies 1 byte, but in PyArrow one boolean element occupies 1 bit so the data can never be shared between them.

In [57]:
np_arr_b = np.array([True, False, True])
np_arr_b

array([ True, False,  True])

In [58]:
pa_arr_b = pa.array(np_arr_b)
pa_arr_b

[
  true,
  false,
  true
]

In [59]:
pa_arr_b.buffers()[1].address == np_arr_b.ctypes.data

False

In [60]:
na.array(pa_arr_b).inspect()

<ArrowArray bool>
- length: 3
- offset: 0
- null_count: 0
- buffers[2]:
  - validity <bool[0 b] >
  - data <bool[1 b] 10100000>
- dictionary: NULL
- children[0]:
